# Domain closeness

Given a variant and a threshold (in %), count how many entries have their **second closest domain within `threshold%` of the closest**.

Source data: the domain neighbours file produced by `tools/classify.py`, which stores for each domain the top-10000 entries and their similarity scores:

`data/semantic_search/<VARIANT>/semantic_search-edition_7-doc2vec-learn-mc_40-ng_1-tm_0.5-ch_sentence.tv2_domains.json`

An entry absent from a domain's list has a score of 0 for that domain (see `helpers/classifiers/semantic_search.py`, `classify`). The metric is **relative**: an entry is counted when `closest > 0` and `second >= closest * (1 - threshold/100)`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))
from helpers import settings

VARIANT = '2026-07-19'
THRESHOLD = 5

DOMAIN_NEIGHBOURS_FILENAME = 'semantic_search-edition_7-doc2vec-learn-mc_40-ng_1-tm_0.5-ch_sentence.tv2_domains.json'

domain_neighbours_path = Path(settings.DATA_PATH, 'semantic_search', VARIANT, DOMAIN_NEIGHBOURS_FILENAME)
if not domain_neighbours_path.exists():
    raise FileNotFoundError(f'No domain neighbours file found for variant "{VARIANT}" at {domain_neighbours_path}')

domain_results = json.loads(domain_neighbours_path.read_text())

print(f'Variant: {VARIANT}')
print(f'Threshold: {THRESHOLD}%')
print(f'Domains: {list(domain_results.keys())}')
print(f'Entries per domain list: {[len(r["aids"]) for r in domain_results.values()]}')

## Build the entry x domain score map

Each domain records its top-10000 entries with their similarity scores. An entry that does not appear in a domain's list has an implicit score of 0 for that domain. We gather, for every entry that appears in at least one list, its non-zero score per domain.

In [ ]:
entry_scores = {}
for domain, results in domain_results.items():
    for aid, score in zip(results['aids'], results['scores']):
        entry_scores.setdefault(aid, {})[domain] = score

print(f'Entries present in at least one domain list: {len(entry_scores)}')

## Compute closest vs second closest domain

For each entry we sort its domain scores descending. `closest` is the highest score, `second` the next one (0 if the entry appears in only one domain list). An entry is **within the threshold** when `closest > 0` and `second >= closest * (1 - threshold/100)`.

In [ ]:
rows = []
factor = 1 - THRESHOLD / 100
for aid, scores in entry_scores.items():
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    closest_domain, closest_score = ordered[0]
    second_domain, second_score = ordered[1] if len(ordered) > 1 else (None, 0.0)
    ratio = second_score / closest_score if closest_score > 0 else 0.0
    within_threshold = closest_score > 0 and second_score >= closest_score * factor
    rows.append({
        'aid': aid,
        'closest_domain': closest_domain,
        'closest_score': closest_score,
        'second_domain': second_domain,
        'second_score': second_score,
        'ratio': ratio,
        'within_threshold': within_threshold,
    })

df = pd.DataFrame(rows).set_index('aid')
df.head()

## Result

Number and percentage of entries whose second closest domain is within the threshold of the closest.

In [ ]:
total = len(df)
count = int(df['within_threshold'].sum())
percentage = count / total * 100 if total else 0

print(f'{count} of {total} entries ({percentage:.1f}%) have their second closest domain within {THRESHOLD}% of the closest')

## Enrich with entry titles

Load the variant index (edition 7 only) and merge titles to inspect the flagged entries.

In [ ]:
index_path = Path(settings.DATA_PATH, VARIANT, 'index.json')
index = pd.read_json(index_path, orient='table')
index = index[index['edition'] != 9][['title']]

flagged = df[df['within_threshold']].join(index, how='inner')
flagged = flagged[['title', 'closest_domain', 'closest_score', 'second_domain', 'second_score', 'ratio']]
flagged.sort_values('ratio', ascending=False).head(20)

## Breakdown by closest domain

How the flagged entries distribute across their closest domain.

In [ ]:
breakdown = flagged.groupby('closest_domain').size().sort_values(ascending=False)
breakdown

## Distribution of second/closest ratios

Histogram of the ratio of the second closest score over the closest score, with the threshold line at `1 - threshold/100`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['ratio'], bins=50)
ax.axvline(factor, color='red', linestyle='--', label=f'threshold = 1 - {THRESHOLD}% = {factor:.2f}')
ax.set_xlabel('second / closest score')
ax.set_ylabel('number of entries')
ax.set_title(f'Domain closeness — {VARIANT}')
ax.legend()
plt.tight_layout()
plt.show()